In [ ]:
!pip install tensorflow
!pip install matplotlib
!pip install ipywidgets


In [ ]:
# Import necessary libraries
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
import io
import ipywidgets as widgets


# Introduction

This lab is designed to introduce the basics of deep learning by interacting with a pre-built model. The workflow includes data preprocessing, viewing the model architecture, and making predictions. The goal is to become familiar with a deep-learning tool without having to build the model from scratch.


In [ ]:
# Load the VGG16 model
model = VGG16(weights='imagenet')

# Display the model architecture
model.summary()


In [ ]:
# Load and preprocess an image
def load_and_preprocess_image(image_path):
    img = load_img(image_path, target_size=(224, 224))
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    return img, img_array


## Sample image

Before running the next cell, upload the test image to the Colab Files panel and name it `sample.jpeg`.


In [ ]:
# Use the image already stored in Colab
image_filename = 'sample.jpeg'

# Load and preprocess the uploaded image
sample_image, processed_image = load_and_preprocess_image(image_filename)

# Display the sample image
plt.imshow(sample_image)
plt.axis('off')
plt.show()


In [ ]:
# Make predictions
predictions = model.predict(processed_image)

# Decode and print the top 3 predictions
decoded_predictions = decode_predictions(predictions, top=3)[0]
print(decoded_predictions)


## Interactive image prediction

Use the upload control below to choose an image, then click **Make Prediction** to see VGG16's top three ImageNet predictions.


In [ ]:
# Upload button to load images
upload = widgets.FileUpload()
display(upload)

# Button to make predictions
predict_button = widgets.Button(description="Make Prediction")
display(predict_button)

# Function to handle button click
def on_click(change):
    if not upload.value:
        print("Upload an image first.")
        return

    if isinstance(upload.value, dict):
        file_info = next(iter(upload.value.values()))
    else:
        file_info = upload.value[0]

    img_data = file_info['content']
    if hasattr(img_data, 'tobytes'):
        img_data = img_data.tobytes()

    img = Image.open(io.BytesIO(img_data)).convert('RGB')
    img = img.resize((224, 224))

    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    predictions = model.predict(img_array)
    decoded_predictions = decode_predictions(predictions, top=3)[0]
    print(decoded_predictions)

predict_button.on_click(on_click)


# Activity 5: Understanding Predictions

The original lab directions ask us to explore how input changes such as rotation and noise affect predictions. The comparison cell below was added so the group can test the **same image** under three conditions:

1. Original image
2. Rotated image
3. Image with visual noise added

The goal is to observe whether VGG16's predictions stay stable when the pixels change, even though the subject remains the same.


In [ ]:
# We use clear_output so that each new experiment replaces
# the previous results instead of piling results on the screen.
from IPython.display import clear_output

# STEP 1 - CREATE THE CONTROLS FOR OUR EXPERIMENT
rotation_slider = widgets.IntSlider(
    value=20,
    min=0,
    max=180,
    step=5,
    description='Rotation:'
)

noise_slider = widgets.IntSlider(
    value=20,
    min=0,
    max=80,
    step=5,
    description='Noise:'
)

compare_button = widgets.Button(
    description='Compare Predictions'
)

comparison_output = widgets.Output()

display(
    rotation_slider,
    noise_slider,
    compare_button,
    comparison_output
)

# STEP 2 - GET THE SAME IMAGE WE UPLOADED ABOVE
def get_uploaded_image():
    if not upload.value:
        raise ValueError("Upload an image in the interactive prediction cell first.")

    if isinstance(upload.value, dict):
        file_info = next(iter(upload.value.values()))
    else:
        file_info = upload.value[0]

    img_data = file_info['content']

    if hasattr(img_data, 'tobytes'):
        img_data = img_data.tobytes()

    img = Image.open(
        io.BytesIO(img_data)
    ).convert('RGB')

    return img.resize((224, 224))

# STEP 3 - CREATE ONE STANDARD PREDICTION PROCESS
def predict_image(img):
    img_array = img_to_array(img)
    img_array = np.expand_dims(
        img_array,
        axis=0
    )
    img_array = preprocess_input(img_array)

    predictions = model.predict(
        img_array,
        verbose=0
    )

    return decode_predictions(
        predictions,
        top=3
    )[0]

# STEP 4 - BUILD THE ACTUAL EXPERIMENT
def run_comparison(change):
    with comparison_output:
        clear_output(wait=True)

        try:
            original = get_uploaded_image()
        except ValueError as error:
            print(error)
            return

        # EXPERIMENT 1: ROTATION
        rotated = original.rotate(
            rotation_slider.value,
            fillcolor='white'
        )

        # EXPERIMENT 2: ADD NOISE
        original_array = np.array(
            original
        ).astype(np.float32)

        rng = np.random.default_rng(42)

        noise = rng.normal(
            0,
            noise_slider.value,
            original_array.shape
        )

        noisy_array = np.clip(
            original_array + noise,
            0,
            255
        ).astype(np.uint8)

        noisy = Image.fromarray(noisy_array)

        # STEP 5 - ORGANIZE OUR THREE TEST CONDITIONS
        versions = [
            ("Original", original),
            (f"Rotated {rotation_slider.value}°", rotated),
            (f"Noise level {noise_slider.value}", noisy)
        ]

        # STEP 6 - DISPLAY THE IMAGES SIDE BY SIDE
        fig, axes = plt.subplots(
            1,
            3,
            figsize=(12, 4)
        )

        results = []

        for ax, (title, image) in zip(axes, versions):
            ax.imshow(image)
            ax.set_title(title)
            ax.axis('off')

            results.append(
                (title, predict_image(image))
            )

        plt.tight_layout()
        plt.show()

        # STEP 7 - COMPARE VGG16'S ANSWERS
        for title, predictions in results:
            print("\n" + title)

            for rank, (_, label, score) in enumerate(
                predictions,
                start=1
            ):
                print(
                    f"{rank}. {label}: {score:.2%}"
                )

# STEP 8 - CONNECT THE BUTTON TO OUR EXPERIMENT
compare_button.on_click(run_comparison)


# Conclusion and Discussion

Reflect on the lab activities. Discuss how the pre-trained model was able to make predictions, the role of data preprocessing, and the impact of input modifications on the model's predictions.
